In [ ]:
import os
import sys
import h5py
import time
import tables
import logging
import warnings
import datetime
import numpy as np
import pandas as pd
import scipy.io as spio
from scipy import signal
from scipy import interpolate

def convertItem2String(element):
    ds = file[element]
    asciiArr = file.get(ds.name).value
    return ''.join(chr(i) for i in asciiArr)

def convertItem2Int(element):
    ds = file[element]
    asciiArr = file.get(ds.name).value
    return int(''.join(chr(i) for i in asciiArr))

def convertItem2Float(element):
    ds = file[element]
    asciiArr = file.get(ds.name).value
    return float(''.join(chr(i) for i in asciiArr))

def getDoNotUse(file):
    doNotUse = (file.get((file.get('/session/doNotUse')).name).value)
    doNotUseFin = []
    for i in range(np.shape(doNotUse)[0]):
        doNotUseFin.append(doNotUse[i][0])
    return doNotUseFin

def getRegionIndex(file):
    regionMap = (file.get(file.get('/session/CLU2RegionMapping').name).value)
    return regionMap[0].astype(int)

def getBehData(file, doNotUse):
    trialwiseCorrect = file.get('/session/trialCorrect')
    correctTrials = []
    for trialNum in range(len(trialwiseCorrect)):
        correctTrials.append((trialwiseCorrect[trialNum])[0])
    behData = file.get('/session/behData/textdata')
    behDataShape = np.shape(behData)
    convBehData = []
    convBehData.append(np.transpose(correctTrials))
    convBehData.append(list(map(convertItem2String,behData[1])))
    convBehData.append(list(map(convertItem2String,behData[4])))
    convBehData.append(list(map(convertItem2String,behData[2])))
    convBehData.append(list(map(convertItem2String,behData[3])))
    convBehData = np.asarray(convBehData,dtype=object).transpose()
    convBehDataFin = []
    for i in range(np.shape(convBehData)[0]):
        tempHolder = []
        for j in range(np.shape(convBehData)[1]):
            tempHolder.append(convBehData[i][j])
        convBehDataFin.append(tempHolder)
    return convBehDataFin

def getRegionIndex(file):
    regionMap = (file.get(file.get('/session/CLU2RegionMapping').name).value)
    return regionMap[0].astype(int)

def getSpikeTimesTrial(file,padToCount):
    sessionSpike = file.get('/session/spike/spikeTimes')
    spikeTimeArr = []
    for trialNum in range(len(sessionSpike)):
        spikeTimesds = file[sessionSpike[trialNum][0]]
        spikeTimes = file.get(spikeTimesds.name).value
        if np.shape(spikeTimes)[0] == 2:
            spikeTimeArr.append([])
        else:
            spikeTimeArr.append(spikeTimes)
    for i in range(padToCount-len(spikeTimeArr)):
        spikeTimeArr.append(None)
    return spikeTimeArr

def getSpikeTimesWP(file,padToCount):
    sessionSpike = file.get('/session/WP/spike/spikeTimes')
    spikeTimeArr = []
    for trialNum in range(len(sessionSpike)):
        spikeTimesds = file[sessionSpike[trialNum][0]]
        spikeTimes = file.get(spikeTimesds.name).value
        if np.shape(spikeTimes)[0] == 2:
            spikeTimeArr.append([])
        else:
            spikeTimeArr.append(spikeTimes)
    for i in range(padToCount-len(spikeTimeArr)):
        spikeTimeArr.append(None)
    return spikeTimeArr

def getSingleTrialTimes(element):
    groupdata = file.get(file[element].name)
    tTrav = []
    for j in range(np.shape(groupdata)[1]):
        tTrav.append(np.take(groupdata,j,axis=1))
    tTrav = np.asarray(tTrav)
    tTravFlat = np.ndarray.flatten(tTrav)
    return tTravFlat

def getTrialTimes(file, doNotUse):
    VTDataLin = file.get('/session/VTdata/TS')
    linearized = []
    for i in range(np.shape(VTDataLin)[0]):
        if doNotUse[i] == 1.:
            linearized.append([])
        else:
            linearized.append(list(map(getSingleTrialTimes,VTDataLin[i])))
    startArmTimes = []
    for sublist in linearized:
        flat_list = []
        for subsublist in sublist:
            for item in subsublist:
                flat_list.append(item)
        startArmTimes.append(flat_list)
    return startArmTimes

def reparseBehData(inputBehData):
    typeTrial = []
    corrTrial = []
    jourTrial = []
    for a,arr in enumerate(inputBehData):
        typeTrial.append(arr[1][0])
        corrTrial.append(int(arr[0]))
        jourTrial.append(f'{arr[3][0]}{arr[4][0]}')
    return typeTrial,corrTrial,jourTrial

def getTrialRawLFPAndRawTimes(file,doNotUse,padToCount):
    sessionLFP = file.get('/session/LFPdata/data')
    rawLFP = []
    for trialNum in range(len(sessionLFP)):
        if doNotUse[trialNum] != 3:
            trialLFPds = file[sessionLFP[trialNum][0]]
            trialLFP = file.get(trialLFPds.name).value
            if np.shape(trialLFP)[0] == 2:
                rawLFP.append([])
            else:
                channel = trialLFP[:]
                rawLFP.append(channel)
    sessionTime = file.get('/session/LFPdata/TS')
    times = []
    for tNum in range(len(sessionTime)):
        if doNotUse[tNum] != 3:
            trialTimeds = file[sessionTime[tNum][0]]
            trialTime = file.get(trialTimeds.name).value
            if np.shape(trialTime) == 2:
                times.append([])
            else:
                channelTimes = (trialTime[:]).flatten()
                times.append(channelTimes)
    for i in range(padToCount-len(rawLFP)):
        rawLFP.append(None)
    for i in range(padToCount-len(times)):
        times.append(None)
    return rawLFP,times

def getWPRawLFPAndRawTimes(file,doNotUse,padToCount):
    sessionLFP = file.get('/session/WP/LFPdata/data')
    rawLFP = []
    for trialNum in range(len(sessionLFP)):
        if doNotUse[trialNum] != 3:
            trialLFPds = file[sessionLFP[trialNum][0]]
            trialLFP = file.get(trialLFPds.name).value
            if np.shape(trialLFP)[0] == 2:
                rawLFP.append([])
            else:
                channel = trialLFP[:]
                rawLFP.append(channel)
    sessionTime = file.get('/session/WP/LFPdata/TS')
    times = []
    for tNum in range(len(sessionTime)):
        if doNotUse[tNum] != 3:
            trialTimeds = file[sessionTime[tNum][0]]
            trialTime = file.get(trialTimeds.name).value
            if np.shape(trialTime) == 2:
                times.append([])
            else:
                channelTimes = (trialTime[:]).flatten()
                times.append(channelTimes)
    for i in range(padToCount-len(rawLFP)):
        rawLFP.append(None)
    for i in range(padToCount-len(times)):
        times.append(None)
    return rawLFP,times

def getSpikeToTetrode(file):
    spikeMap = (file.get(file.get('/session/CLU2ColMapping').name).value)
    return spikeMap[0].astype(int)-1

def quickParseKey(inputKeyValues,ignoreList):
    outputKeyValues = [None]
    for a,values in enumerate(inputKeyValues):
        if a not in ignoreList:
            outputKeyValues.append(values)
    return outputKeyValues[1:]

def getCueIndices(inputKey):
    cueInds = []
    for tr,val in enumerate(inputKey):
        if val == 'C':
            cueInds.append(tr)
    return cueInds

def filterDict(inputRawDict):
    outFilteredDict = {'regionIndex':inputRawDict['regionIndex'].flatten(),
                       'spikeRefLFP':inputRawDict['spikeRefLFP'].flatten(),
                      }
    deleteList = np.nonzero(inputRawDict['doNotUse'])[0].flatten()
    for ikey in inputRawDict.keys():
        if ikey != 'regionIndex' and ikey != 'doNotUse' and ikey != 'spikeRefLFP':
            if ikey == 'trialType':
                outFilteredDict['cueIndices'] = getCueIndices(inputRawDict[ikey])
            else:
                outFilteredDict[ikey] = quickParseKey(inputRawDict[ikey],deleteList)
    return outFilteredDict

def parseOneTrial(refTimes,refVoltage,refSpike):
    currTimeRef = np.arange(refTimes[0],refTimes[-1]+5000,5000)
    currRefInds = [np.argmax(a>refTimes) for a in currTimeRef]
    outSpikes = 0
    for unitRow in refSpike:
        unitHisto,unitBins = np.histogram(unitRow.flatten(),bins=currTimeRef,density=False)
        unitHistoUpd = np.where(unitHisto>1,1,unitHisto)
        unitHistoRes = np.reshape(np.concatenate((unitHistoUpd,[0]),axis=0),(1,len(currTimeRef)))
        if np.shape(outSpikes) == ():
            outSpikes = unitHistoRes
        else:
            outSpikes = np.concatenate((outSpikes,unitHistoRes),axis=0)
    return outSpikes,np.take(refVoltage,currRefInds,axis=1)

def buildSpatialAndCue(inputValueDict):
    cueIndex = inputValueDict['cueIndices']
    outDict = {'regionIndex':inputValueDict['regionIndex'].astype(int).flatten()}
    cueSpikes = 0
    spaSpikes = 0
    cueSignal = 0
    spaSignal = 0
    for tr in range(len(inputValueDict['wpSignal'])):
        check = len(np.arange(inputValueDict['wpSignalTimes'][tr][0],inputValueDict['wpSignalTimes'][tr][-1]+5000,5000))
        currTimes = np.concatenate((inputValueDict['wpSignalTimes'][tr].flatten(),
                                    inputValueDict['trSignalTimes'][tr].flatten()),axis=0)
        currVoltage = np.concatenate((inputValueDict['wpSignal'][tr],inputValueDict['trSignal'][tr]),axis=1)
        currSpike = np.nan_to_num(np.concatenate((inputValueDict['wpSpikeTimes'][tr],
                                                  inputValueDict['trSpikeTimes'][tr]),axis=1),
                                  nan=-10,posinf=-10,neginf=-10)
        trialSpike,trialVolt = parseOneTrial(currTimes,currVoltage,currSpike)
        if tr in cueIndex:
            if np.shape(cueSpikes) == ():
                cueSignal = trialVolt
                cueSpikes = trialSpike
            else:
                cueSignal = np.concatenate((cueSignal,trialVolt),axis=1)
                cueSpikes = np.concatenate((cueSpikes,trialSpike),axis=1)
        else:
            if np.shape(spaSpikes) == ():
                spaSignal = trialVolt
                spaSpikes = trialSpike
            else:
                spaSignal = np.concatenate((spaSignal,trialVolt),axis=1)
                spaSpikes = np.concatenate((spaSpikes,trialSpike),axis=1)
    try:
        cueSignalRef = np.shape(cueSignal)[0]
        spikeRefRefLFP = np.where(inputValueDict['spikeRefLFP']>=cueSignalRef,0,inputValueDict['spikeRefLFP'])
        outDict.update({'cueSpike':cueSpikes.astype(int),'cueLFP':np.take(cueSignal,spikeRefRefLFP,axis=0),
                        'spaSpike':spaSpikes.astype(int),'spaLFP':np.take(spaSignal,spikeRefRefLFP,axis=0),
                       })
    except:
        try:
            outDict.update({'cueSpike':cueSpikes.astype(int),'cueLFP':[],
                            'spaSpike':spaSpikes.astype(int),'spaLFP':[],
                           })
        except:
            outDict.update({'cueSpike':cueSpikes,'cueLFP':[],
                            'spaSpike':spaSpikes.astype(int),'spaLFP':[],
                           })
    return outDict

def digestData(file):
    doNotUse = getDoNotUse(file)
    behData = getBehData(file, doNotUse)
    trialType,trialCorrect,trialJourney = reparseBehData(behData)
    regionIndex = getRegionIndex(file)
    spikeTetrodeRef = getSpikeToTetrode(file)
    trialSpikeArr = getSpikeTimesTrial(file,len(doNotUse))
    wpSpikeArr = getSpikeTimesWP(file,len(doNotUse))
    trialLFP,trialTimes = getTrialRawLFPAndRawTimes(file,doNotUse,len(doNotUse))
    wpLFP,wpTimes = getWPRawLFPAndRawTimes(file,doNotUse,len(doNotUse))
    saveDict = {'regionIndex':regionIndex,
                'spikeRefLFP':spikeTetrodeRef,
                'doNotUse':doNotUse,
                'trialType':trialType,
                'wpSpikeTimes':wpSpikeArr,
                'trSpikeTimes':trialSpikeArr,
                'wpSignalTimes':wpTimes,
                'wpSignal':wpLFP,
                'trSignalTimes':trialTimes,
                'trSignal':trialLFP,
               }
    filteredDict = filterDict(saveDict)
    spaCueDict = buildSpatialAndCue(filteredDict)
    return spaCueDict

warnings.filterwarnings('ignore')
doneFiles = []
testPath = ''
savePath = ''
for aRoot,aDirs,aFiles in os.walk(testPath):
    for aFile in aFiles:
        if aFile.endswith('.mat') and aFile not in doneFiles:
            print(aFile)
            currRawFile = os.path.join(testPath,aFile)
            currSavFile = os.path.join(savePath,aFile)
            file = h5py.File(currRawFile, 'r')
            dictToSave = digestData(file)
            saveVar = spio.savemat(currSavFile,dictToSave)
            print(aFile)
print('Done')
